# 04 — Explore visually

Explore the project's three story angles before building the publication
charts, using the shared Pillow templates. Charts render inline via `display()`.

- **A. Domestic** — adjusted vs nominal gross (dumbbell)
- **B. Worldwide** — domestic vs international split (dumbbell)
- **C. Genre** — which genres skew international vs domestic (diverging bars)

> Note: this project renders with the shared **Pillow** factory rather than
> matplotlib. (matplotlib does not run in this project's Python 3.14 venv — a
> known `MarkerStyle` deepcopy recursion during axis-tick rendering — and the
> publication path is Pillow anyway, so matplotlib isn't a dependency here.)

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import pandas as pd, duckdb
from src.ingest import load_config
from chart_templates import lollipop, single_ranked_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: this notebook only reads, so it runs even if another notebook
# kernel has the DuckDB file open (DuckDB is single-writer).
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
films = con.execute('SELECT * FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15').df()
films['label'] = films['title'] + '  (' + films['release_year'].astype(str) + ')'
img_w, img_h, _ = PRESETS['twitter_landscape']
def money(v):
    return f'${abs(v)/1e9:.2f}B' if abs(v) >= 1e9 else f'${abs(v)/1e6:.0f}M'
films[['title','adjusted_gross','nominal_gross','release_year']].head()

## A. Domestic — adjusted vs nominal
The gap between what a film made at the time (gold) and its inflation-adjusted
gross (teal). Domestic (U.S. & Canada) only.

In [ ]:
display(lollipop(
    films, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='Adjusted vs nominal domestic gross - top 15',
    subtitle='Teal = adjusted (2022 $), gold = nominal (release $)',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted', value2_label='Nominal', img_width=img_w, img_height=img_h,
))

## B. Worldwide — domestic vs international
The reason the domestic chart is only half the story: for most top films the
international (rest-of-world) take dwarfs the domestic one. Gold = domestic,
teal = international. Watch **Ne Zha 2** — a Chinese blockbuster with a huge
international total and almost no domestic gross.

In [ ]:
# Restrict to US-made films so 'domestic' (US & Canada) = the film's home market.
ww = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE)
    SELECT w.title, w.release_year, w.domestic_gross, w.foreign_gross, w.worldwide_gross
    FROM films_worldwide w JOIN us ON us.title=w.title AND us.release_year=w.release_year
    ORDER BY w.worldwide_gross DESC LIMIT 15''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
display(lollipop(
    ww, category_col='label', value_col='foreign_gross', value2_col='domestic_gross',
    label_col='worldwide_gross', value_fmt=money,
    title='Hollywood films: home (US/Canada) vs abroad (top 15)',
    subtitle='US-made films only. Gold = home (US & Canada), teal = rest of world; label = worldwide total.',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Rest of world', value2_label='Home', img_width=img_w, img_height=img_h,
))

## C. Share of box office earned abroad, by genre
Plain international share per genre (US-made films): of everything the genre
earned worldwide, what fraction came from outside the U.S. & Canada. Sorted
high to low. Each film's gross is attributed to all its genres (so grosses
double-count across genres — fine for a share, see SOURCES.md). The story:
every genre earns most of its money abroad; sci-fi is simply the lowest.

In [ ]:
genre = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE)
    SELECT gl.genre AS category, COUNT(*) n,
      ROUND(100.0*SUM(w.foreign_gross)/(SUM(w.domestic_gross)+SUM(w.foreign_gross)),1) AS value
    FROM films_worldwide w
    JOIN us ON us.title=w.title AND us.release_year=w.release_year
    JOIN film_genres_long gl ON gl.title=w.title AND gl.release_year=w.release_year
    GROUP BY gl.genre HAVING COUNT(*)>=10 ORDER BY value DESC''').df()
genre['pct_label'] = genre['value'].apply(lambda v: f'{v:.0f}%')
display(single_ranked_bars(genre, category_col='category', value_col='value',
    total_label_col='pct_label', bar_color='#005F73',
    title='Every blockbuster genre earns most of its money abroad',
    subtitle='Share of worldwide box office earned outside the U.S. & Canada, by genre (top US-made films).',
    img_width=img_w, img_height=img_h))

## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode,
other notebooks).

In [ ]:
con.close()
print('connection closed')

---
**Next:** `06-viz-social.ipynb` builds the publication versions of these three
charts (full titling/source) and saves them to `outputs/social/`.